# 02 Transform - Synapse PySpark Medallion Pipeline

This notebook reads raw taxi parquet data from Azure Data Lake, writes a Bronze Delta layer, cleans and enriches records into Silver, and produces a Gold aggregate for revenue by pickup location.

## Imports and Spark setup

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

from src.paths import BRONZE_PATH, GOLD_PATH, RAW_PATH, SILVER_PATH
from src.transform import add_time_features, aggregate_revenue_by_location, clean_taxi_data

builder = (
    SparkSession.builder.appName("nyc-taxi-transform")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Reading raw data from: {RAW_PATH}")
print(f"Bronze path: {BRONZE_PATH}")
print(f"Silver path: {SILVER_PATH}")
print(f"Gold path: {GOLD_PATH}")

## Read source data with schema evolution enabled

In [ ]:
raw_df = (
    spark.read.format("parquet")
    .option("mergeSchema", "true")
    .load(RAW_PATH)
)

raw_df.printSchema()
raw_df.show(5, truncate=False)

## Write Bronze layer

In [ ]:
(
    raw_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_PATH)
)

bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze record count: {bronze_df.count()}")

## Clean and engineer Silver layer

In [ ]:
silver_df = add_time_features(clean_taxi_data(bronze_df))

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Silver record count: {silver_df.count()}")
silver_df.select("fare_amount", "trip_distance", "PULocationID", "DOLocationID", "hour").show(5, truncate=False)

## Build Gold aggregate - revenue per location

In [ ]:
gold_df = aggregate_revenue_by_location(silver_df)

(
    gold_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)

gold_df.show(20, truncate=False)